In [ ]:
import os

sys.path.append('/mnt/c/Users/Isabella/tcc')
DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger1'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger1_gt.txt')
output_dir = "resultados_video/"
video_path = "resultados_video/tracking_output.mp4"

In [ ]:
import cv2
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt
import json

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# Primeiro frame

In [ ]:
# Carrega primeiro frame  # lembrar de carregar certo em tons de cinza
first_frame = cv2.imread(image_paths[0])
print(f"Primeiro frame carregado: {first_frame.shape}")

first_gt = all_ground_truths[0]
print(f"Ground Truth do primeiro frame: {first_gt}")

# Mostra o frame com a bbox
first_frame_with_bbox = first_frame.copy()
x, y, w, h = map(int, first_gt)
cv2.rectangle(first_frame_with_bbox, (x, y), (x + w, y + h), (0, 255, 0), 2) 

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(first_frame_with_bbox, cv2.COLOR_BGR2RGB))
plt.title('Primeiro Frame - Objeto a Rastrear')
plt.axis('off')
plt.show()

In [ ]:
# --- PASSO 2: EXTRAIR E PRÉ-PROCESSAR PATCH ---
print("\n Extraindo e pré-processando patch...")

# Extrai patch do objeto
patch = tracker_utils.extract_patch(first_frame, first_gt)

# Mostra o patch ORIGINAL (antes do pré-processamento)
plt.figure(figsize=(8, 6))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
plt.title(f'Patch Original\n shape:{patch.shape}')
plt.axis('off')


In [ ]:
# def preprocess_patch(patch):
#     # Converte para grayscale caso seja RGB
#     if len(patch.shape) == 3:
#         gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
#     else:
#         gray = patch.copy()

#     # 1) MÉDIA GLOBAL  (correção do erro)
#     global_mean = int(np.mean(gray))
#     no_bg_global = cv2.subtract(gray, global_mean)

#     # 2) Binarização Otsu
#     _, otsu = cv2.threshold(
#         gray, 0, 255,
#         cv2.THRESH_BINARY + cv2.THRESH_OTSU
#     )

#     # 3) Retornar vetor flatten binário
#     vector = (otsu.flatten() > 0).astype(np.uint8)

#     return vector
import cv2
import numpy as np
import matplotlib.pyplot as plt

def preprocess_patch(patch, show=True):
    if len(patch.shape) == 3:
        gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    else:
        gray = patch.copy()

    global_mean = int(np.mean(gray))
    no_bg_global = cv2.subtract(gray, global_mean)
    # Otsu adaptativo
    otsu_local = cv2.adaptiveThreshold(
        no_bg_global, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        3, 3
    )

    # Gradiente
    gx = cv2.Sobel(no_bg_global, cv2.CV_16S, 1, 0)
    gy = cv2.Sobel(no_bg_global, cv2.CV_16S, 0, 1)
    grad = cv2.convertScaleAbs(gx) + cv2.convertScaleAbs(gy)

    _, grad_bin = cv2.threshold(
        grad, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Visualização
    if show:
        plt.figure(figsize=(10, 3))

        plt.subplot(1, 4, 1)
        plt.imshow(gray, cmap="gray")
        plt.title("Gray")
        plt.axis("off")

        plt.subplot(1, 4, 3)
        plt.imshow(otsu_local, cmap="gray")
        plt.title("Otsu local")
        plt.axis("off")

        plt.subplot(1, 4, 4)
        plt.imshow(grad_bin, cmap="gray")
        plt.title("Gradiente")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
first_pattern = preprocess_patch(patch)
print(f"Primeiro Pattern pré-processado: {first_pattern.shape}")

In [ ]:
# Parâmetros
parameters = {
    # "INPUT_PATTERN_SIDE": 32,                  
    "CLUSWISARD_ADDRESS_SIZE":7,          
    "MAX_RETRAIN_SCORE": 0.5,       
    # "MIN_RETRAIN_SCORE": 0.4,    
    "CLUSWISARD_MIN_SCORE": 0.5,               
    "CLUSWISARD_THRESHOLD": 1,                 
    "CLUSWISARD_DISCRIMINATOR_LIMIT": 100,      
    "CLUSWISARD_BLEACHING_ACTIVATED": True,     
    "CLUSWISARD_ACTIVATION_DEGREE": True,
    "CLUSWISARD_RETURN_CONFIDENCE": True,
    "CLUSWISARD_CLASSES_DEGREES": True,
    "MAX_SEARCH_WINDOW_SCALE": 0.2,
    "STEP_SIZE": 20,
}

clus = ClusWisard(
            parameters['CLUSWISARD_ADDRESS_SIZE'], # address size
            parameters['CLUSWISARD_MIN_SCORE'], # min score
            parameters['CLUSWISARD_THRESHOLD'], # threshold
            parameters['CLUSWISARD_DISCRIMINATOR_LIMIT'], # discriminator limit
            bleachingActivated=parameters['CLUSWISARD_BLEACHING_ACTIVATED'],
            returnActivationDegree=parameters['CLUSWISARD_ACTIVATION_DEGREE'],
            returnConfidence=parameters['CLUSWISARD_RETURN_CONFIDENCE'],
            returnClassesDegrees=parameters['CLUSWISARD_CLASSES_DEGREES']
        )


In [ ]:
# Treina com o patch do objeto
clus.train([first_pattern.tolist()], ["object"])
print("ClusWisard treinado com o patch do primeiro frame!")

In [ ]:
print("\n Testando autorreconhecimento...")

# Classifica o MESMO patch usado no treinamento
result = clus.classify([first_pattern.tolist()])[0]
activation = result.get("activationDegree", -1)

print(f"Ativação com o próprio patch de treinamento: {activation}")
print(result)

# Região de Busca

In [ ]:
def generate_search_regions_circular(
    prev_bbox, 
    frame_shape, 
    search_region_scale=1, 
    step_size=0.5,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior,
    onde step_size define o deslocamento em pixels reais
    (1 px = step_size=1).
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2

    raio_max = (max(w, h) / 2) * search_region_scale

    yield (x, y, w, h)  # primeiro retorna o bbox original

    # passo radial em pixels
    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio

        # define espaçamento angular de forma que os pontos na borda
        # fiquem separados por ~step_size pixels ao longo da circunferência
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))
        direction = -1 if clockwise else 1

        thetas = start_angle + direction * np.linspace(0, 2 * np.pi, num_steps, endpoint=False)
        
        # deslocamentos em pixels
        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        # garante que o bbox não ultrapasse os limites do frame
        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)


### Gerando regiões para o primeiro frame

In [ ]:
# Parâmetros
search_window_scale = parameters['MAX_SEARCH_WINDOW_SCALE']
step_size = parameters['STEP_SIZE']          # quanto menor -> mais granular

search_gen = generate_search_regions_circular(
    prev_bbox = (16.0, 30.0, 34.0, 39.0), #TIGER2
    # prev_bbox = (122.0, 58.0, 75.0, 97.0), # DAVID
    # prev_bbox = (142.0, 62.0, 62.0, 98.0),
    frame_shape = first_frame.shape,
    search_region_scale = search_window_scale,
    step_size = step_size
)

### Classificação das regiões de busca do primeiro frame

In [ ]:
# # --- CLASSIFICAR CADA SEARCH REGION ---
# def classify_search_regions(clus, regions, frame):
#     results = []
#     for i, region in enumerate(regions):
#         # Extrai o patch
#         patch_region = tracker_utils.extract_patch(frame, region)
        
#         if patch_region.size == 0:
#             print(f"Region {i}: Patch vazio - ignorando")
#             continue
            
#         # Pré-processa
#         pattern_region = preprocess_patch(patch_region)
        
#         # Classifica com ClusWisard
#         result = clus.classify([pattern_region.tolist()])[0]
#         activation = result.get("activationDegree", -1)
        
#         results.append({
#             'region_id': i,
#             'bbox': region,
#             'activation': activation
#         })
        
#         print(f"Region {i}: ativação = {activation:.4f}")

#     # Ordena por ativação (maior primeiro)
#     results_sorted = sorted(results, key=lambda x: x['activation'], reverse=True)

#     print("\n RANKING DAS SEARCH REGIONS:")
#     for i, result in enumerate(results_sorted):
#         print(f"{i+1}º - Region {result['region_id']}: ativação = {result['activation']:.4f}")

#     # Mostra a melhor região
#     best_region = results_sorted[0]
#     print(f"\n MELHOR REGIÃO: Region {best_region['region_id']} com ativação {best_region['activation']:.4f}")
#     return results_sorted

In [ ]:
# results_sorted = classify_search_regions(
#     clus,
#     search_gen,
#     first_frame
# )

In [ ]:
# def show_top_regions_grid(frame, results_sorted, top_n=5, cols=3):
#     """
#     Mostra as N regiões mais bem classificadas em um grid.

#     Parâmetros:
#         frame: imagem original (BGR)
#         results_sorted: lista retornada de classify_search_regions()
#         top_n: número de regiões a exibir
#         cols: número de colunas no grid
#     """
#     top_n = min(top_n, len(results_sorted))
#     rows = (top_n + cols - 1) // cols

#     plt.figure(figsize=(4 * cols, 4 * rows))

#     h_frame, w_frame = frame.shape[:2]

#     for i in range(top_n):
#         region_info = results_sorted[i]
#         region_id = region_info["region_id"]
#         activation = region_info["activation"]

#         # ---  Converte coordenadas para inteiros
#         x, y, w, h = [int(round(v)) for v in region_info["bbox"]]

#         # --- Garante que os índices estão dentro dos limites do frame
#         x = max(0, min(x, w_frame - 1))
#         y = max(0, min(y, h_frame - 1))
#         w = max(1, min(w, w_frame - x))
#         h = max(1, min(h, h_frame - y))

#         # --- Extrai o patch da região
#         patch = frame[y:y+h, x:x+w]
#         if patch.size == 0:
#             print(f"Region {region_id} vazia - ignorando exibição")
#             continue

#         patch_rgb = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)

#         # --- Mostra patch no grid
#         plt.subplot(rows, cols, i + 1)
#         plt.imshow(patch_rgb)
#         # plt.title(f"#{i+1} Reg {region_id}\nAct={activation:.3f}")
#         plt.axis("off")

#     plt.tight_layout()
#     plt.show()


In [ ]:
# show_top_regions_grid(first_frame, results_sorted)

### Vídeo exemplo de região de busca para o primeiro frame

In [ ]:
# import cv2
# import numpy as np

# max_regions = 300
# output_path = "search_circular.mp4"
# fps = 10
# accumulate = True

# h, w_img = first_frame.shape[:2]
# fourcc = cv2.VideoWriter_fourcc(*"mp4v")
# video_writer = cv2.VideoWriter(output_path, fourcc, fps, (w_img, h))
# print("Aberto?", video_writer.isOpened())

# drawn_regions = []

# for i, region in enumerate(search_gen):
#     if i >= max_regions:
#         break

#     frame_copy = first_frame.copy()

#     # Garante que todas as dimensões estão iguais
#     if frame_copy.shape[:2] != (h, w_img):
#         frame_copy = cv2.resize(frame_copy, (w_img, h))

#     if accumulate:
#         for (px, py, pw, ph) in drawn_regions:
#             cv2.rectangle(frame_copy, (int(px), int(py)), (int(px+pw), int(py+ph)), (0, 0, 255), 1)

#     x_r, y_r, w_r, h_r = map(int, region)
#     cv2.rectangle(frame_copy, (x_r, y_r), (x_r + w_r, y_r + h_r), (0, 255, 0), 2)
#     cv2.putText(frame_copy, f"Region {i}", (x_r, max(0, y_r - 10)),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

#     video_writer.write(cv2.cvtColor(frame_copy, cv2.COLOR_RGB2BGR))
#     drawn_regions.append((x_r, y_r, w_r, h_r))

# video_writer.release()
# print(f"Vídeo salvo em: {output_path} (frames: {len(drawn_regions)})")


# Tracker

In [ ]:
from collections import deque
import cv2
import numpy as np
import matplotlib.pyplot as plt
# from wisardpkg import ClusWisard

# ============================================================
# PARÂMETROS
# ============================================================
CLUSWISARD_DISCRIMINATOR_LIMIT = parameters["CLUSWISARD_DISCRIMINATOR_LIMIT"]
CLUSWISARD_MIN_SCORE = parameters["CLUSWISARD_MIN_SCORE"]

BACKGROUND_RETRAIN_THRESHOLD = 0.7

# ============================================================
# FILA FIFO PARA RETREINO DO FUNDO
# ============================================================
discriminator_queue = deque(maxlen=CLUSWISARD_DISCRIMINATOR_LIMIT)

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================
def has_overlap(b1, b2):
    """
    Retorna True se dois bounding boxes se sobrepõem.
    """
    x1, y1, w1, h1 = map(int, b1)
    x2, y2, w2, h2 = map(int, b2)

    xa = max(x1, x2)
    ya = max(y1, y2)
    xb = min(x1 + w1, x2 + w2)
    yb = min(y1 + h1, y2 + h2)

    return xb > xa and yb > ya


def extract_background_patches(frame, gt_bbox, step_size, patch_size):
    """
    Varre TODA a imagem em grade regular e extrai patches
    que NÃO se sobrepõem ao bounding box do objeto (gt).

    ✔ Varredura full-frame
    ✔ Exclusão apenas pela região do objeto
    """
    H, W, _ = frame.shape
    pw, ph = map(int, patch_size)
    step_size = int(step_size)

    gx, gy, gw, gh = map(int, gt_bbox)
    gt_box = (gx, gy, gw, gh)

    patches = []

    # varredura explícita de toda a imagem
    for y in range(0, H - ph + 1, step_size):
        for x in range(0, W - pw + 1, step_size):

            candidate = (x, y, pw, ph)

            # exclui SOMENTE a região do objeto
            if has_overlap(candidate, gt_box):
                continue

            patch = tracker_utils.extract_patch(frame, candidate)
            if patch is not None and patch.size > 0:
                patches.append(patch)

    return patches


def plot_patches_grid(patches, title="", max_patches=365, cols=5):
    if not patches:
        print(f"Nenhum patch para plotar: {title}")
        return

    n = min(len(patches), max_patches)
    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(cols * 2.2, rows * 2.2))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(patches[i])
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# ============================================================
# MODELO DE FUNDO
# ============================================================
def new_clus_background():
    return ClusWisard(
        parameters["CLUSWISARD_ADDRESS_SIZE"],
        parameters["CLUSWISARD_MIN_SCORE"],
        parameters["CLUSWISARD_THRESHOLD"],
        parameters["CLUSWISARD_DISCRIMINATOR_LIMIT"],
        bleachingActivated=parameters["CLUSWISARD_BLEACHING_ACTIVATED"],
        returnActivationDegree=True,
        returnConfidence=False,
        returnClassesDegrees=False,
    )


# ============================================================
# FRAME INICIAL
# ============================================================
first_frame = cv2.imread(image_paths[0])
first_frame = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

first_gt = tuple(map(int, first_gt))

# ============================================================
# TREINO INICIAL DO FUNDO (FULL FRAME)
# ============================================================
clus_background = new_clus_background()

bg_patches = extract_background_patches(
    frame=first_frame,
    gt_bbox=first_gt,
    step_size=parameters["STEP_SIZE"],
    patch_size=(first_gt[2], first_gt[3]),
)

plot_patches_grid(
    bg_patches,
    title="Patches de Treinamento do Fundo (Varredura Full-Frame)",
)

X_bg, y_bg = [], []
for p in bg_patches:
    pat = preprocess_patch(p)
    X_bg.append(pat.tolist())
    y_bg.append("background")

clus_background.train(X_bg, y_bg)

# ============================================================
# LOOP PRINCIPAL
# ============================================================
prev_bbox = first_gt
all_predictions = [prev_bbox]

selected_patches = []
bg_scores = []

for frame_idx, frame_path in enumerate(image_paths[1:462], start=1):

    frame = cv2.imread(frame_path)
    if frame is None:
        all_predictions.append(prev_bbox)
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    search_gen = generate_search_regions_circular(
        prev_bbox=prev_bbox,
        frame_shape=frame.shape,
        search_region_scale=parameters["MAX_SEARCH_WINDOW_SCALE"],
        step_size=10,
    )

    results = []

    for region in search_gen:
        region = tuple(map(int, region))
        patch = tracker_utils.extract_patch(frame, region)
        if patch is None or patch.size == 0:
            continue

        pattern = preprocess_patch(patch)
        res_bg = clus_background.classify([pattern.tolist()])[0]

        results.append({
            "bbox": region,
            "activation_bg": res_bg.get("activationDegree", 1.0),
            "patch_raw": patch,
            "patch_proc": pattern,
        })

    if not results:
        all_predictions.append(prev_bbox)
        continue

    # objeto = pior score de fundo
    best_region = min(results, key=lambda r: r["activation_bg"])

    prev_bbox = best_region["bbox"]
    all_predictions.append(prev_bbox)

    bg_score = best_region["activation_bg"]
    selected_patches.append(best_region["patch_raw"])
    bg_scores.append(bg_score)

    print(
        f"Frame {frame_idx} | "
        f"act_bg={bg_score:.3f} | "
        f"object_score={1.0 - bg_score:.3f}"
    )

# ============================================================
# VISUALIZA PATCHES DO TRACKING
# ============================================================
plot_patches_grid(
    selected_patches,
    title="Patches Escolhidos pelo Tracker",
)

# ============================================================
# FINAL
# ============================================================
print("\n✅ Tracking concluído!")
print(f"🔁 Total de retreinos do fundo: {len(discriminator_queue)}")


In [ ]:
len(retrain_patches)

In [ ]:
##### import cv2
import os
import shutil

# --- Pasta de saída ---
# Apaga a pasta se existir
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# --- Configuração do vídeo ---
fps = 10  # ajuste se seu vídeo original tiver outro FPS

# Lê o primeiro frame para pegar o tamanho
first_frame = cv2.imread(image_paths[1])
h, w = first_frame.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(video_path, fourcc, fps, (w, h))

# --- Processar e gravar vídeo ---
for frame_idx, frame_path in enumerate(image_paths[1:364], start=1):

    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"Frame {frame_idx} não carregado!")
        continue

    # Pega a bbox prevista correspondente
    if frame_idx < len(all_predictions):
        x, y, bw, bh = map(int, all_predictions[frame_idx])
        cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"Frame {frame_idx}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
        )

    # Escreve no vídeo
    video_writer.write(frame)

video_writer.release()
print(f" Vídeo salvo com sucesso em: {video_path}")


In [ ]:
print(len(X_bg))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def bbox_center(bbox):
    """
    bbox: (x, y, w, h)
    retorna: (cx, cy)
    """
    x, y, w, h = bbox
    return np.array([x + w / 2.0, y + h / 2.0])


errors = []
frames = []

for i in range(0, len(all_ground_truths), 5):

    frame_idx = i
    gt_bbox = all_ground_truths[i]

    result_bbox = all_predictions[i]

    gt_center = bbox_center(gt_bbox)
    res_center = bbox_center(result_bbox)

    error = np.linalg.norm(gt_center - res_center)

    errors.append(error)
    frames.append(frame_idx)

# --- PLOT ---
plt.figure()
plt.plot(frames, errors, linestyle='--')
plt.xlabel("Frame")
plt.ylabel("Erro de centro (pixels)")
plt.title("Erro entre centro do GT e centro do tracker")
plt.grid(True)
plt.show()

media = sum(errors) / len(errors)
print(media)


1- Reproduzir a arquitetura 1 para o vídeo do David
2- Testar o método de subtrair gradativamente o número de retreinos no tempo
3- Testar o variação do threshold
4- Testar limitar o retreino completo
5- Olhar o material (trânsito) pra pegar a subtração de um frame pro outro

In [ ]:
testar um sample de cluswisards:

cria a primeira instancia -> treina até discriminator n vezes (se score abaixo de retrain)
cria a segunda instância -> treina até discriminator n vezes ((se score abaixo de retrain (verificar para a primeira (preferencia do patch escolhido pela primeira) e para a segunda)
cria a terceira instância -> 